# Phase B: merge of PMS and CRM

Input: the two clean files written by Phase A (`pms_cleaning.ipynb` and `crm_cleaning.ipynb`). For each step: what it does, why, and the rows before and after.

- **CHECK** cells show the numbers the next decision is based on. They do not change the data.
- The two systems describe the same bookings, so here we can finally use one file to check and complete the other.
- **Same rules as Phase A:** before changing a value we create a flag column; a fill that is not proven is an ESTIMATE; no invented data.
- The notebook lives in `Project/merge`, reads from `Project/data/processed` and writes the merged table there.

In [1]:
import pandas as pd

## 1. Load
The clean files of Phase A. `booking_id` (PMS) and `BookingRef` (CRM) are the booking code: before merging we check that each one identifies a single row, otherwise the merge would multiply rows.

In [2]:
pms = pd.read_csv("../data/processed/pms_clean.csv", parse_dates=["record_updated_at"])
crm = pd.read_csv("../data/processed/crm_clean.csv", parse_dates=["checkin_date", "reservation_status_date"])
print("PMS:", pms.shape, "| booking_id unique:", pms["booking_id"].is_unique)
print("CRM:", crm.shape, "| BookingRef unique:", crm["BookingRef"].is_unique)

PMS: (24250, 28) | booking_id unique: True
CRM: (24500, 25) | BookingRef unique: True


## 2. Prepare the merge

### Which columns mean the same thing
The two files use different names. Shared information:

| PMS | CRM | Note |
|---|---|---|
| `booking_id` | `BookingRef` | merge key |
| `hotel` | `property` | same two hotels |
| `arrival_date_year` + `_month` + `_day_of_month` | `checkin_date` | arrival date |
| `adults` | `n_adults` | number of adults |
| `is_canceled` | `reservation_status` | Canceled / No-Show = cancelled, Check-Out = stayed |
| `adr` × nights | `total_price` | price of the stay |
| weekend + week nights | `reservation_status_date` − `checkin_date` | length of stay, only for Check-Out |

### Key and source columns
We give the key the same name in both files, so the merge can use `on`. No other column name is shared, so no suffixes are needed.

We build the PMS arrival date as a real date now, while its three columns are still integers (after an outer merge they would become floats).

We also add `in_pms` and `in_crm`: after an outer merge they tell which file each row comes from.

In [3]:
crm = crm.rename(columns={"BookingRef": "booking_id"})
print("shared column names:", set(pms.columns) & set(crm.columns))

pms["arrival_date"] = pd.to_datetime(pms["arrival_date_year"].astype(str) + "-"
                                     + pms["arrival_date_month"].astype(str) + "-"
                                     + pms["arrival_date_day_of_month"].astype(str), errors="coerce")
print("PMS arrival dates not readable:", pms["arrival_date"].isna().sum())

pms["in_pms"] = True
crm["in_crm"] = True
keys_pms = set(pms["booking_id"])
keys_crm = set(crm["booking_id"])
print("CHECK | in both:", len(keys_pms & keys_crm),
      "| only PMS:", len(keys_pms - keys_crm), "| only CRM:", len(keys_crm - keys_pms))

shared column names: {'booking_id'}
PMS arrival dates not readable: 0
CHECK | in both: 23750 | only PMS: 500 | only CRM: 750


## 3. Merge

**Decision: outer merge.** 23,750 bookings are in both files, 500 only in the PMS and 750 only in the CRM. Together they are the "about 25,000 bookings" of the brief, which also says that some bookings appear in only one file.
- An inner merge would keep 23,750 rows and lose 1,250 real bookings.
- A left or right merge would lose the bookings of one of the two files.
- The outer merge keeps one row per booking. The columns of the missing file are NaN.

Expected rows: 23,750 + 500 + 750 = 25,000. We check the shape before and after.

In [4]:
print("rows before: PMS", len(pms), "| CRM", len(crm))
df = pd.merge(pms, crm, on="booking_id", how="outer")
df["in_pms"] = df["in_pms"].notna()
df["in_crm"] = df["in_crm"].notna()
print("rows after:", len(df), "| columns:", df.shape[1], "| booking_id unique:", df["booking_id"].is_unique)
pd.crosstab(df["in_pms"], df["in_crm"])

rows before: PMS 24250 | CRM 24500
rows after: 25000 | columns: 55 | booking_id unique: True


in_crm,False,True
in_pms,,
False,0,750
True,500,23750


### Columns that changed type
With the outer merge, integer columns of one file get NaN on the rows of the other file, so pandas turns them into floats. The flag columns of Phase A become NaN for the rows that were not in that file: there the flag was never checked, so `False` (not touched) is the right value.

In [5]:
flags = ["meal_undefined", "adr_invalid", "adr_outlier", "adr_zero", "week_nights_missing", "zero_nights",
         "no_guests", "children_missing",
         "segment_undefined", "channel_undefined", "country_placeholder", "country_cn_ambiguous",
         "lead_time_missing", "adults_over_10", "date_inconsistent"]
print("NaN in the flags before:", df[flags].isna().sum().sum())
for col in flags:
    df[col] = df[col] == True
print("NaN in the flags after:", df[flags].isna().sum().sum())
df[flags].sum()

NaN in the flags before: 9500
NaN in the flags after: 0


meal_undefined          236
adr_invalid              70
adr_outlier               1
adr_zero                377
week_nights_missing      35
zero_nights             147
no_guests                31
children_missing        809
segment_undefined         2
channel_undefined         3
country_placeholder     114
country_cn_ambiguous    228
lead_time_missing        25
adults_over_10            2
date_inconsistent        88
dtype: int64

## 4. Shared columns: one value per booking

For each shared piece of information we keep **one** column. The PMS is the reference: it is the hotel management system and the Phase A checks showed it is the more complete file. The CRM fills it only where the PMS has no value, i.e. the 750 bookings that are only in the CRM.

Conflicts between the two files, where both have a value and they differ, are handled in section 5: here no PMS value is changed.

### 4.1 Hotel

In [6]:
both = df["in_pms"] & df["in_crm"]
print("CHECK | hotel different from property, rows in both files:", (both & (df["hotel"] != df["property"])).sum())
print("hotel missing before:", df["hotel"].isna().sum())
df["hotel"] = df["hotel"].fillna(df["property"])
print("hotel missing after:", df["hotel"].isna().sum())
df["hotel"].value_counts()

CHECK | hotel different from property, rows in both files: 0
hotel missing before: 750
hotel missing after: 0


hotel
City Hotel      16621
Resort Hotel     8379
Name: count, dtype: int64

### 4.2 Arrival date
The PMS date built in section 2 is completed with `checkin_date` for the CRM-only rows. The three PMS columns stay for now: the conflicts are checked in section 5.

In [7]:
print("arrival_date missing before:", df["arrival_date"].isna().sum())
df["arrival_date"] = df["arrival_date"].fillna(df["checkin_date"])
print("arrival_date missing after:", df["arrival_date"].isna().sum(),
      "| from", df["arrival_date"].min().date(), "to", df["arrival_date"].max().date())

arrival_date missing before: 750
arrival_date missing after: 0 | from 2015-07-01 to 2017-08-31


### 4.3 Adults
`adults` is complete in the PMS; `n_adults` has 244 NaN in the CRM. We complete `adults` with `n_adults` on the CRM-only rows.

In [8]:
print("adults missing before:", df["adults"].isna().sum())
df["adults"] = df["adults"].fillna(df["n_adults"])
print("adults missing after:", df["adults"].isna().sum(),
      "| of which CRM-only rows with n_adults missing:", (~df["in_pms"] & df["n_adults"].isna()).sum())

adults missing before: 750
adults missing after: 8 | of which CRM-only rows with n_adults missing: 8


### 4.4 Cancellation
`reservation_status` says the same thing as `is_canceled`. The crosstab on the rows in both files shows the rule: Canceled and No-Show are 1, Check-Out is 0. The only exceptions are 50 rows, handled in section 5.

In [9]:
status_code = df["reservation_status"].map({"Canceled": 1, "No-Show": 1, "Check-Out": 0})
print(pd.crosstab(df["is_canceled"][both], df["reservation_status"][both]))
print("is_canceled missing before:", df["is_canceled"].isna().sum())
df["is_canceled"] = df["is_canceled"].fillna(status_code)
print("is_canceled missing after:", df["is_canceled"].isna().sum())

reservation_status  Canceled  Check-Out  No-Show
is_canceled                                     
0.0                       50      14841        0
1.0                     8610          0      249
is_canceled missing before: 750
is_canceled missing after: 0


### 4.5 What the outer merge leaves missing
For the 500 PMS-only rows all CRM columns are NaN, and for the 750 CRM-only rows the PMS columns without a CRM counterpart (meal, room types, children, adr, ...) are NaN. These values are not in either file, so they stay missing: nothing is invented.

In [10]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

company_id                        23624
agent_id                           3800
children                           1559
meal                               1469
total_price                         990
adr                                 940
stays_in_week_nights                785
arrival_date_year                   750
arrival_date_month                  750
babies                              750
required_car_parking_spaces         750
assigned_room_type                  750
booking_changes                     750
deposit_type                        750
record_updated_at                   750
stays_in_weekend_nights             750
reserved_room_type                  750
arrival_date_day_of_month           750
total_of_special_requests           750
n_adults                            744
guest_country                       614
lead_time_days                      525
distribution_channel                503
market_segment                      502
is_repeated_guest                   500


## 5. Conflicts between the two files

Rows in both files where the PMS and the CRM give different values for the same information. The course rules (Sessions 3 and 4):
- assessment finds a conflict, it does not choose;
- **correct only with evidence of the cause**, otherwise keep the value and flag it ("a guess dressed as a fix is worse than a flag").

So for each conflict we look for evidence in a third column (usually the price, which links nights and adr), and we change a value only when the evidence points to one file.

### 5.1 Arrival date
45 bookings have a different arrival date in the two files. For the Check-Out rows we can test which date is right: departure (CRM status date) minus arrival must give the nights of the PMS.

In [11]:
nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
arrival_diff = both & (df["arrival_date"] != df["checkin_date"])
checkout = df["reservation_status"] == "Check-Out"
print("arrival conflicts:", arrival_diff.sum(),
      "| days CRM - PMS:", (df["checkin_date"] - df["arrival_date"])[arrival_diff].dt.days.value_counts().to_dict())
print("CHECK | Check-Out among them:", (arrival_diff & checkout).sum(),
      "| departure - PMS arrival = PMS nights:",
      (arrival_diff & checkout & ((df["reservation_status_date"] - df["arrival_date"]).dt.days == nights)).sum(),
      "| departure - CRM check-in = PMS nights:",
      (arrival_diff & checkout & ((df["reservation_status_date"] - df["checkin_date"]).dt.days == nights)).sum())

arrival conflicts: 45 | days CRM - PMS: {1: 16, -1: 15, 7: 14}
CHECK | Check-Out among them: 30 | departure - PMS arrival = PMS nights: 30 | departure - CRM check-in = PMS nights: 0


**Decision:** the PMS date is right in every row we can test, so `arrival_date` keeps the PMS value (it already does). Flag `arrival_conflict` on the 45 rows; no value changes.

In [12]:
df["arrival_conflict"] = arrival_diff
print("arrival_conflict:", df["arrival_conflict"].sum(), "| rows:", len(df))

arrival_conflict: 45 | rows: 25000


### 5.2 Adults
75 bookings have a different number of adults. They are of two kinds:
- (a) the PMS says more than 10 adults and the CRM 1 or 2;
- (b) the two files differ by one adult.

The price is the independent evidence: a room for 55 people does not cost the same as a room for 2.

In [13]:
adults_diff = both & df["n_adults"].notna() & (df["adults"] != df["n_adults"])
typo = adults_diff & (df["adults"] > 10) & (df["n_adults"] <= 10)
print("adults conflicts:", adults_diff.sum(), "| (a) PMS > 10, CRM <= 10:", typo.sum(),
      "| (b) others:", (adults_diff & ~typo).sum())
print("CHECK (a) | PMS values:", df["adults"][typo].value_counts().to_dict(),
      "| CRM values:", df["n_adults"][typo].value_counts().to_dict())
print("   adr of these rows: min", df["adr"][typo].min(), "max", df["adr"][typo].max(),
      "| median adr by adults (PMS):", df[df["adults"] <= 4].groupby("adults")["adr"].median().to_dict())
print("   reserved room types:", df["reserved_room_type"][typo].value_counts().to_dict())
print("   more than 10 adults confirmed by the CRM:", (both & (df["adults"] > 10) & (df["adults"] == df["n_adults"])).sum())
pd.crosstab(df["adults"][adults_diff & ~typo], df["n_adults"][adults_diff & ~typo])

adults conflicts: 75 | (a) PMS > 10, CRM <= 10: 19 | (b) others: 56
CHECK (a) | PMS values: {55.0: 7, 26.0: 6, 20.0: 3, 50.0: 2, 40.0: 1} | CRM values: {2.0: 15, 1.0: 4}
   adr of these rows: min 34.0 max 157.0 | median adr by adults (PMS): {0.0: 9.0, 1.0: 80.0, 2.0: 95.0, 3.0: 151.75, 4.0: 192.86}
   reserved room types: {'A': 16, 'E': 2, 'D': 1}
   more than 10 adults confirmed by the CRM: 2


n_adults,1.0,2.0,3.0,4.0
adults,,,,
1.0,0,2,0,0
2.0,35,0,15,0
3.0,0,3,0,1


**Decision (a): correct.** The PMS values come from a small fixed set (20, 26, 40, 50, 55), the rooms are mostly type A and the price is that of 1-2 people. The CRM and the price agree, so this is a known mistake of the PMS: `adults` takes the CRM value, flag `adults_from_crm`. The 2 real groups (26 and 55 adults) are confirmed by the CRM and stay.

**Decision (b): flag only.** A difference of one adult has no evidence either way (the price is closer to the PMS value in 29 cases and to the CRM in 27). The PMS value stays, flag `adults_conflict`.

The rows with more than 10 adults in the PMS and no CRM value cannot be checked: flag `adults_unverified`.

In [14]:
df["adults_from_crm"] = typo
df.loc[typo, "adults"] = df.loc[typo, "n_adults"]
df["adults_conflict"] = adults_diff & ~typo
df["adults_unverified"] = (df["adults"] > 10) & df["n_adults"].isna()
print("adults_from_crm:", df["adults_from_crm"].sum(), "| adults_conflict:", df["adults_conflict"].sum(),
      "| adults_unverified:", df["adults_unverified"].sum())
print("adults > 10 after:", (df["adults"] > 10).sum(), "| rows:", len(df))

adults_from_crm: 19 | adults_conflict: 56 | adults_unverified: 2
adults > 10 after: 4 | rows: 25000


### 5.3 Cancellation
50 bookings are not cancelled in the PMS but `Canceled` in the CRM. They are exactly the 50 `date_inconsistent` rows "Canceled with a status date after check-in". If the guest really stayed, the status date should be the departure day: arrival plus nights.

In [15]:
status_diff = both & (df["is_canceled"] == 0) & (df["reservation_status"] == "Canceled")
print("status conflicts:", status_diff.sum(), "| also date_inconsistent:", (status_diff & df["date_inconsistent"]).sum())
print("CHECK | status date = arrival + PMS nights:",
      (status_diff & ((df["reservation_status_date"] - df["arrival_date"]).dt.days == nights)).sum())

status conflicts: 50 | also date_inconsistent: 50
CHECK | status date = arrival + PMS nights: 50


**Decision: correct.** In all 50 rows the status date is exactly the departure day, so the guest stayed and the CRM label is wrong. `is_canceled` stays 0, `reservation_status` becomes `Check-Out`, flag `status_from_pms`. This way the two columns no longer contradict each other.

In [16]:
df["status_from_pms"] = status_diff
df.loc[status_diff, "reservation_status"] = "Check-Out"
print("status_from_pms:", df["status_from_pms"].sum())
pd.crosstab(df["is_canceled"][both], df["reservation_status"][both])

status_from_pms: 50


reservation_status,Canceled,Check-Out,No-Show
is_canceled,,,
0.0,0,14891,0
1.0,8610,0,249


### 5.4 Nights
For Check-Out rows, departure minus arrival must give the PMS nights. Where it does not, the price decides: `total_price / adr` gives the nights the guest paid for.

In [17]:
checkout = df["reservation_status"] == "Check-Out"
stay = (df["reservation_status_date"] - df["arrival_date"]).dt.days
nights_diff = both & checkout & nights.notna() & (stay != nights)
paid_nights = (df["total_price"] / df["adr"]).round()
testable = nights_diff & df["total_price"].notna() & (df["adr"] > 0)
print("nights conflicts:", nights_diff.sum(), "| also date_inconsistent:", (nights_diff & df["date_inconsistent"]).sum())
print("CHECK | testable on the price:", testable.sum(), "| price confirms PMS nights:", (testable & (paid_nights == nights)).sum())

nights conflicts: 35 | also date_inconsistent: 30
CHECK | testable on the price: 32 | price confirms PMS nights: 31


**Decision: keep the PMS, flag.** The price confirms the PMS nights in almost every testable row, so the CRM status date is the wrong value. The nights stay, flag `nights_conflict`: the CRM dates of these rows are not used to rebuild anything.

In [18]:
df["nights_conflict"] = nights_diff
print("nights_conflict:", df["nights_conflict"].sum(), "| rows:", len(df))

nights_conflict: 35 | rows: 25000


## 6. Recompute the missing values

Prices and nights are linked by one formula, `total_price = adr × nights`, which holds within 1 euro on 22,436 of the 22,708 rows where all three are known. So when one of the three is missing, the other two give it back. Before using the formula we check the rows where it does not hold (6.1), because a wrong price would spread into the recomputed values.

### 6.1 Prices that do not match
272 rows break the formula. We look at the direction and size of the gap, and at the typical adr of the same hotel, room type, year and month, computed only on the rows that agree.

In [19]:
nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
testable = both & df["total_price"].notna() & (df["adr"] > 0) & (nights > 0)
price_diff = testable & ((df["total_price"] - df["adr"] * nights).abs() > 1)
ratio = df["total_price"] / (df["adr"] * nights)
print("testable rows:", testable.sum(), "| formula broken:", price_diff.sum())
print("CHECK | total_price / (adr x nights): min", ratio[price_diff].min().round(2), "max", ratio[price_diff].max().round(2),
      "| rows where the CRM price is lower:", (price_diff & (ratio < 1)).sum())

keys = ["hotel", "reserved_room_type", "arrival_date_year", "arrival_date_month"]
df["adr_agree"] = df["adr"].where(testable & ~price_diff)
typical = df.groupby(keys)["adr_agree"].transform("median")
df = df.drop(columns="adr_agree")
crm_adr = df["total_price"] / nights
print("CHECK | median of adr / typical adr | PMS adr:", (df["adr"] / typical)[price_diff].median().round(3),
      "| adr from the CRM price:", (crm_adr / typical)[price_diff].median().round(3),
      "| rows that agree (control):", (df["adr"] / typical)[testable & ~price_diff].median().round(3))
print("      PMS adr closer to the typical adr:", ((df["adr"] - typical).abs() < (crm_adr - typical).abs())[price_diff].sum(),
      "of", price_diff.sum())

testable rows: 22708 | formula broken: 272
CHECK | total_price / (adr x nights): min 1.15 max 1.6 | rows where the CRM price is lower: 0
CHECK | median of adr / typical adr | PMS adr: 0.987 | adr from the CRM price: 1.344 | rows that agree (control): 1.0
      PMS adr closer to the typical adr: 200 of 272


**Decision: correct the CRM price.** The gap always goes the same way: the CRM price is 15% to 60% higher, never lower. A random typing error would go both ways. The PMS adr matches the typical price of the same room in the same month (median ratio 0.99, like the rows that agree), while the adr implied by the CRM price is 34% above it. So the CRM price is inflated and the PMS adr is right: `total_price` becomes adr × nights, flag `price_inflated`.

In [20]:
df["price_inflated"] = price_diff
df.loc[price_diff, "total_price"] = (df["adr"] * nights)[price_diff].round(2)
print("price_inflated:", df["price_inflated"].sum(),
      "| formula broken after:", (testable & ((df["total_price"] - df["adr"] * nights).abs() > 1)).sum())

price_inflated: 272 | formula broken after: 0


### 6.2 Week nights
The 35 rows with `stays_in_week_nights` missing (code -1 in the raw PMS). Two sources, in order of reliability:
1. **CRM dates:** for Check-Out rows without a date problem, departure minus arrival gives the total nights; minus the weekend nights, the week nights. Flag `week_nights_from_crm`.
2. **Price:** `total_price / adr` gives the nights paid; minus the weekend nights, the week nights. Used only when the result is a whole number. It is an **ESTIMATE**: flag `week_nights_estimated`.

In [21]:
week_missing = df["in_pms"] & df["stays_in_week_nights"].isna()
stay = (df["reservation_status_date"] - df["arrival_date"]).dt.days
from_dates = week_missing & (df["reservation_status"] == "Check-Out") & ~df["date_inconsistent"] & stay.notna()
paid = df["total_price"] / df["adr"]
from_price = week_missing & ~from_dates & (df["adr"] > 0) & df["total_price"].notna() & ((paid - paid.round()).abs() < 0.01)
print("week nights missing:", week_missing.sum(), "| from the dates:", from_dates.sum(),
      "| from the price:", from_price.sum(), "| no source:", (week_missing & ~from_dates & ~from_price).sum())

week nights missing: 35 | from the dates: 18 | from the price: 15 | no source: 2


In [22]:
df["week_nights_from_crm"] = from_dates
df.loc[from_dates, "stays_in_week_nights"] = (stay - df["stays_in_weekend_nights"])[from_dates]
df["week_nights_estimated"] = from_price
df.loc[from_price, "stays_in_week_nights"] = (paid.round() - df["stays_in_weekend_nights"])[from_price]
print("rebuilt values:", df.loc[from_dates | from_price, "stays_in_week_nights"].value_counts().sort_index().to_dict())
print("week nights still missing:", df["stays_in_week_nights"][df["in_pms"]].isna().sum())

rebuilt values: {0.0: 1, 1.0: 10, 2.0: 11, 3.0: 4, 4.0: 3, 5.0: 2, 6.0: 2}
week nights still missing: 2


### 6.3 adr
The adr missing in the PMS (set to NaN in Phase A because negative or above 1,000, or missing from the start) is recomputed as `total_price / nights`. Flag `adr_from_crm`. For 0-night stays the formula cannot be used: every 0-night row with a known adr has adr 0, so these become 0, flag `adr_estimated` (ESTIMATE).

In [23]:
nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
adr_missing = df["in_pms"] & df["adr"].isna()
print("CHECK | adr of 0-night rows with a known adr:", df["adr"][nights == 0].value_counts().to_dict())
from_price = adr_missing & df["total_price"].notna() & (nights > 0)
zero = adr_missing & (nights == 0)
print("adr missing:", adr_missing.sum(), "| from the price:", from_price.sum(), "| 0 nights:", zero.sum(),
      "| no source:", (adr_missing & ~from_price & ~zero).sum())

CHECK | adr of 0-night rows with a known adr: {0.0: 145}
adr missing: 190 | from the price: 177 | 0 nights: 3 | no source: 10


In [24]:
df["adr_from_crm"] = from_price
df.loc[from_price, "adr"] = (df["total_price"] / nights)[from_price].round(2)
df["adr_estimated"] = zero
df.loc[zero, "adr"] = 0
print("adr still missing (PMS rows):", df["adr"][df["in_pms"]].isna().sum())

adr still missing (PMS rows): 10


### 6.4 total_price
The CRM price missing in rows that are also in the PMS is recomputed as adr × nights. Flag `price_from_pms`. The 500 PMS-only rows are left as they are: they have no CRM price at all, and the analyses can use adr.

In [25]:
price_missing = both & df["total_price"].isna()
from_adr = price_missing & df["adr"].notna() & nights.notna()
print("total_price missing:", price_missing.sum(), "| from adr x nights:", from_adr.sum())
df["price_from_pms"] = from_adr
df.loc[from_adr, "total_price"] = (df["adr"] * nights)[from_adr].round(2)
print("total_price still missing (rows in both files):", df["total_price"][both].isna().sum())

total_price missing: 469 | from adr x nights: 466
total_price still missing (rows in both files): 3


## 7. Export
We drop the columns that are now duplicates:
- `property`: identical to `hotel` in every row in both files, and already used to fill it;
- `checkin_date`: already used to fill `arrival_date`; where the two differ, the CRM date was shown to be wrong (5.1).

`n_adults` stays: in the 56 `adults_conflict` rows nothing proves which file is right, so the CRM value is kept next to the PMS one. Final order by booking code, then the shape and how many rows each flag marks.

In [26]:
df = df.drop(columns=["property", "checkin_date"])
df = df.sort_values("booking_id").reset_index(drop=True)
print("rows, columns:", df.shape, "| booking_id unique:", df["booking_id"].is_unique)
flags = ["arrival_conflict", "adults_from_crm", "adults_conflict", "adults_unverified", "status_from_pms",
         "nights_conflict", "price_inflated", "week_nights_from_crm", "week_nights_estimated",
         "adr_from_crm", "adr_estimated", "price_from_pms"]
df[flags].sum()

rows, columns: (25000, 65) | booking_id unique: True


arrival_conflict          45
adults_from_crm           19
adults_conflict           56
adults_unverified          2
status_from_pms           50
nights_conflict           35
price_inflated           272
week_nights_from_crm      18
week_nights_estimated     15
adr_from_crm             177
adr_estimated              3
price_from_pms           466
dtype: int64

In [27]:
df.to_csv("../data/processed/merged_clean.csv", index=False)
print("saved ../data/processed/merged_clean.csv:", df.shape)

saved ../data/processed/merged_clean.csv: (25000, 65)
